# Multi-LogiEval Results Analysis: CoT vs Lean vs Two-Stage

This notebook analyzes three approaches across all depths (d1-d5) and all logic types (FOL, NM, PL).

**Dataset**: 150 questions (3 logic types × 5 depths × 10 samples)
- **FOL**: First-order logic
- **NM**: Non-monotonic logic
- **PL**: Propositional logic
- **Depths**: d1 (1 rule) through d5 (5 chained rules)

**Research Question**: How do different approaches perform across reasoning depths and logic types?

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load Results

In [ ]:
# Load all three result sets
with open('../results/multilogieval/all_depths/cot/all_results.json', 'r') as f:
    cot_results = json.load(f)

with open('../results/multilogieval/all_depths/lean/all_results.json', 'r') as f:
    lean_results = json.load(f)

with open('../results/multilogieval/all_depths/two_stage/all_results.json', 'r') as f:
    two_stage_results = json.load(f)

print(f"Loaded {len(cot_results)} questions from each approach")
print(f"\nOverall Accuracies:")
print(f"  CoT:       {sum(1 for r in cot_results if r['correct'])}/150 = {sum(1 for r in cot_results if r['correct'])/150*100:.2f}%")
print(f"  Lean:      {sum(1 for r in lean_results if r['correct'])}/150 = {sum(1 for r in lean_results if r['correct'])/150*100:.2f}%")
print(f"  Two-Stage: {sum(1 for r in two_stage_results if r['correct'])}/150 = {sum(1 for r in two_stage_results if r['correct'])/150*100:.2f}%")

## 2. Overall Performance Comparison

In [ ]:
# Calculate overall accuracy
cot_acc = sum(1 for r in cot_results if r['correct']) / len(cot_results) * 100
lean_acc = sum(1 for r in lean_results if r['correct']) / len(lean_results) * 100
two_stage_acc = sum(1 for r in two_stage_results if r['correct']) / len(two_stage_results) * 100

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
approaches = ['CoT', 'Lean', 'Two-Stage']
accuracies = [cot_acc, lean_acc, two_stage_acc]
colors = ['#2ecc71', '#3498db', '#9b59b6']

bars = ax1.bar(approaches, accuracies, color=colors)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('Overall Accuracy Comparison', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 100)
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{acc:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Correct/Incorrect breakdown
cot_correct = sum(1 for r in cot_results if r['correct'])
lean_correct = sum(1 for r in lean_results if r['correct'])
two_stage_correct = sum(1 for r in two_stage_results if r['correct'])

x = np.arange(3)
correct_counts = [cot_correct, lean_correct, two_stage_correct]
incorrect_counts = [150 - c for c in correct_counts]

ax2.bar(x, correct_counts, label='Correct', color='#2ecc71')
ax2.bar(x, incorrect_counts, bottom=correct_counts, label='Incorrect', color='#e74c3c')
ax2.set_xticks(x)
ax2.set_xticklabels(approaches)
ax2.set_ylabel('Number of Questions', fontsize=12)
ax2.set_title('Correct vs Incorrect Breakdown', fontsize=14, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("KEY OBSERVATION")
print("="*70)
if lean_acc > cot_acc:
    print(f"✓ Lean ({lean_acc:.1f}%) outperforms CoT ({cot_acc:.1f}%) by {lean_acc - cot_acc:.1f}%")
else:
    print(f"✗ CoT ({cot_acc:.1f}%) outperforms Lean ({lean_acc:.1f}%) by {cot_acc - lean_acc:.1f}%")
    print("  → Different from FOLIO results!")
print("="*70)

## 3. Performance by Logic Type

In [ ]:
# Calculate accuracy by logic type
logic_types = ['fol', 'nm', 'pl']
logic_names = {'fol': 'FOL', 'nm': 'NM', 'pl': 'PL'}

cot_by_logic = {}
lean_by_logic = {}
two_stage_by_logic = {}

for logic in logic_types:
    cot_logic = [r for r in cot_results if r['logic_type'] == logic]
    lean_logic = [r for r in lean_results if r['logic_type'] == logic]
    two_stage_logic = [r for r in two_stage_results if r['logic_type'] == logic]
    
    cot_by_logic[logic] = sum(1 for r in cot_logic if r['correct']) / len(cot_logic) * 100
    lean_by_logic[logic] = sum(1 for r in lean_logic if r['correct']) / len(lean_logic) * 100
    two_stage_by_logic[logic] = sum(1 for r in two_stage_logic if r['correct']) / len(two_stage_logic) * 100

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(logic_types))
width = 0.25

ax.bar(x - width, [cot_by_logic[l] for l in logic_types], width, label='CoT', color='#2ecc71')
ax.bar(x, [lean_by_logic[l] for l in logic_types], width, label='Lean', color='#3498db')
ax.bar(x + width, [two_stage_by_logic[l] for l in logic_types], width, label='Two-Stage', color='#9b59b6')

ax.set_xlabel('Logic Type', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Accuracy by Logic Type', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([logic_names[l] for l in logic_types])
ax.legend()
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print table
print("\n" + "="*70)
print("ACCURACY BY LOGIC TYPE")
print("="*70)
print(f"{'Logic':<8} {'CoT':<12} {'Lean':<12} {'Two-Stage':<12}")
print("-"*70)
for logic in logic_types:
    print(f"{logic_names[logic]:<8} {cot_by_logic[logic]:>6.1f}%     {lean_by_logic[logic]:>6.1f}%     {two_stage_by_logic[logic]:>6.1f}%")
print("="*70)

## 4. Performance by Depth (Reasoning Chain Length)

In [ ]:
# Calculate accuracy by depth
depths = ['d1_Data', 'd2_Data', 'd3_Data', 'd4_Data', 'd5_Data']
depth_labels = ['d1', 'd2', 'd3', 'd4', 'd5']

cot_by_depth = {}
lean_by_depth = {}
two_stage_by_depth = {}

for depth in depths:
    cot_depth = [r for r in cot_results if r['depth_dir'] == depth]
    lean_depth = [r for r in lean_results if r['depth_dir'] == depth]
    two_stage_depth = [r for r in two_stage_results if r['depth_dir'] == depth]
    
    cot_by_depth[depth] = sum(1 for r in cot_depth if r['correct']) / len(cot_depth) * 100
    lean_by_depth[depth] = sum(1 for r in lean_depth if r['correct']) / len(lean_depth) * 100
    two_stage_by_depth[depth] = sum(1 for r in two_stage_depth if r['correct']) / len(two_stage_depth) * 100

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(depths))
ax.plot(x, [cot_by_depth[d] for d in depths], 'o-', label='CoT', color='#2ecc71', linewidth=2, markersize=8)
ax.plot(x, [lean_by_depth[d] for d in depths], 's-', label='Lean', color='#3498db', linewidth=2, markersize=8)
ax.plot(x, [two_stage_by_depth[d] for d in depths], '^-', label='Two-Stage', color='#9b59b6', linewidth=2, markersize=8)

ax.set_xlabel('Reasoning Depth (Number of Chained Rules)', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Accuracy by Reasoning Depth', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(depth_labels)
ax.legend(fontsize=11)
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Print table
print("\n" + "="*70)
print("ACCURACY BY DEPTH")
print("="*70)
print(f"{'Depth':<8} {'CoT':<12} {'Lean':<12} {'Two-Stage':<12}")
print("-"*70)
for depth, label in zip(depths, depth_labels):
    print(f"{label:<8} {cot_by_depth[depth]:>6.1f}%     {lean_by_depth[depth]:>6.1f}%     {two_stage_by_depth[depth]:>6.1f}%")
print("="*70)

# Analyze degradation
cot_d1 = cot_by_depth['d1_Data']
cot_d5 = cot_by_depth['d5_Data']
lean_d1 = lean_by_depth['d1_Data']
lean_d5 = lean_by_depth['d5_Data']
ts_d1 = two_stage_by_depth['d1_Data']
ts_d5 = two_stage_by_depth['d5_Data']

print(f"\nPerformance Degradation (d1 → d5):")
print(f"  CoT:       {cot_d1:.1f}% → {cot_d5:.1f}% (Δ = {cot_d5 - cot_d1:+.1f}%)")
print(f"  Lean:      {lean_d1:.1f}% → {lean_d5:.1f}% (Δ = {lean_d5 - lean_d1:+.1f}%)")
print(f"  Two-Stage: {ts_d1:.1f}% → {ts_d5:.1f}% (Δ = {ts_d5 - ts_d1:+.1f}%)")

## 5. Heatmap: Logic Type × Depth

In [ ]:
# Create accuracy matrices for heatmaps
def create_accuracy_matrix(results):
    matrix = np.zeros((len(logic_types), len(depths)))
    for i, logic in enumerate(logic_types):
        for j, depth in enumerate(depths):
            subset = [r for r in results if r['logic_type'] == logic and r['depth_dir'] == depth]
            if subset:
                matrix[i, j] = sum(1 for r in subset if r['correct']) / len(subset) * 100
    return matrix

cot_matrix = create_accuracy_matrix(cot_results)
lean_matrix = create_accuracy_matrix(lean_results)
two_stage_matrix = create_accuracy_matrix(two_stage_results)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, matrix, title, cmap in zip(axes, 
                                     [cot_matrix, lean_matrix, two_stage_matrix],
                                     ['CoT', 'Lean', 'Two-Stage'],
                                     ['Greens', 'Blues', 'Purples']):
    im = ax.imshow(matrix, cmap=cmap, aspect='auto', vmin=0, vmax=100)
    ax.set_xticks(np.arange(len(depth_labels)))
    ax.set_yticks(np.arange(len(logic_types)))
    ax.set_xticklabels(depth_labels)
    ax.set_yticklabels([logic_names[l] for l in logic_types])
    ax.set_xlabel('Depth', fontsize=11)
    ax.set_ylabel('Logic Type', fontsize=11)
    ax.set_title(f'{title} Accuracy (%)', fontsize=12, fontweight='bold')
    
    # Add text annotations
    for i in range(len(logic_types)):
        for j in range(len(depths)):
            text = ax.text(j, i, f'{matrix[i, j]:.0f}',
                          ha='center', va='center', color='white' if matrix[i, j] < 50 else 'black',
                          fontweight='bold')
    
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

## 6. Lean Verification Statistics

In [ ]:
# Extract Lean verification stats
lean_verified = sum(1 for r in lean_results if r.get('lean_verification', {}).get('success', False))
lean_iterations = [r.get('num_iterations', r.get('iterations', 1)) for r in lean_results if 'lean_code' in r]

two_stage_verified = 0
two_stage_iterations = []
for r in two_stage_results:
    if 'all_cycles' in r and r['all_cycles']:
        final_cycle = r['all_cycles'][-1]
        if final_cycle.get('lean_success', False):
            two_stage_verified += 1
        # Count total Lean iterations
        total_iters = 0
        for cycle in r['all_cycles']:
            if 'stage2_iterations' in cycle:
                total_iters += len(cycle['stage2_iterations'])
        if total_iters > 0:
            two_stage_iterations.append(total_iters)

print("="*70)
print("LEAN VERIFICATION STATISTICS")
print("="*70)
print(f"\nLean Simple:")
print(f"  Verification success: {lean_verified}/150 ({lean_verified/150*100:.1f}%)")
print(f"  Avg iterations: {np.mean(lean_iterations):.2f}")
print(f"  Iteration distribution: {Counter(lean_iterations)}")

print(f"\nTwo-Stage:")
print(f"  Verification success: {two_stage_verified}/150 ({two_stage_verified/150*100:.1f}%)")
if two_stage_iterations:
    print(f"  Avg Lean iterations: {np.mean(two_stage_iterations):.2f}")
    print(f"  Iteration distribution: {Counter(two_stage_iterations)}")
print("="*70)

# Visualize iteration distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Lean
iter_counts = Counter(lean_iterations)
ax1.bar(iter_counts.keys(), iter_counts.values(), color='#3498db')
ax1.set_xlabel('Number of Iterations', fontsize=11)
ax1.set_ylabel('Frequency', fontsize=11)
ax1.set_title('Lean: Iteration Distribution', fontsize=12, fontweight='bold')
ax1.set_xticks(sorted(iter_counts.keys()))

# Two-stage
if two_stage_iterations:
    iter_counts = Counter(two_stage_iterations)
    ax2.bar(iter_counts.keys(), iter_counts.values(), color='#9b59b6')
    ax2.set_xlabel('Total Lean Iterations', fontsize=11)
    ax2.set_ylabel('Frequency', fontsize=11)
    ax2.set_title('Two-Stage: Lean Iteration Distribution', fontsize=12, fontweight='bold')
    if iter_counts:
        ax2.set_xticks(sorted(iter_counts.keys()))

plt.tight_layout()
plt.show()

## 7. Key Insights and Comparison with FOLIO

In [ ]:
print("="*70)
print("KEY INSIGHTS: MULTI-LOGIEVAL")
print("="*70)

print(f"\n1. OVERALL PERFORMANCE:")
if lean_acc > cot_acc:
    print(f"   ✓ Lean ({lean_acc:.1f}%) > CoT ({cot_acc:.1f}%) by {lean_acc - cot_acc:.1f}%")
    print(f"   → Lean verification HELPS on Multi-LogiEval")
else:
    print(f"   ✗ CoT ({cot_acc:.1f}%) > Lean ({lean_acc:.1f}%) by {cot_acc - lean_acc:.1f}%")
    print(f"   → Lean verification HURTS on Multi-LogiEval")

print(f"\n2. BY LOGIC TYPE:")
for logic in logic_types:
    cot_val = cot_by_logic[logic]
    lean_val = lean_by_logic[logic]
    ts_val = two_stage_by_logic[logic]
    best = max(cot_val, lean_val, ts_val)
    best_name = 'CoT' if best == cot_val else ('Lean' if best == lean_val else 'Two-Stage')
    print(f"   {logic_names[logic]}: {best_name} wins ({best:.1f}%) - CoT: {cot_val:.1f}%, Lean: {lean_val:.1f}%, TS: {ts_val:.1f}%")

print(f"\n3. DEPTH DEGRADATION:")
print(f"   CoT:       {cot_d5 - cot_d1:+.1f}% (d1 → d5)")
print(f"   Lean:      {lean_d5 - lean_d1:+.1f}% (d1 → d5)")
print(f"   Two-Stage: {ts_d5 - ts_d1:+.1f}% (d1 → d5)")
if abs(lean_d5 - lean_d1) < abs(cot_d5 - cot_d1):
    print(f"   → Lean shows LESS degradation than CoT")
else:
    print(f"   → CoT shows LESS degradation than Lean")

print(f"\n4. COMPARISON WITH FOLIO:")
print(f"   FOLIO:          CoT (85.7%) > Two-Stage (79.3%) > Lean (74.9%)")
print(f"   Multi-LogiEval: Lean ({lean_acc:.1f}%) {'>' if lean_acc > cot_acc else '<'} CoT ({cot_acc:.1f}%) {'>' if cot_acc > two_stage_acc else '<'} Two-Stage ({two_stage_acc:.1f}%)")
if (lean_acc > cot_acc) != (74.9 > 85.7):  # Different ordering
    print(f"   → DIFFERENT ORDERING! Dataset characteristics matter.")

print(f"\n5. LEAN VERIFICATION:")
print(f"   Success rate: {lean_verified/150*100:.1f}% (Lean), {two_stage_verified/150*100:.1f}% (Two-Stage)")
print(f"   → High verification rates across both approaches")

print("="*70)